# MMMU-val 최종 실행 (Qwen3-VL-4B-Instruct, 16k, seed 3407)

이 노트북은 **3단계**로 진행한다. 각 단계 셀은 독립적으로 Drive를 마운트하고 경로를 정의하므로, 런타임 유형을 바꿔도(CPU ↔ A100) 그대로 실행할 수 있다.

| 단계 | 런타임 | 하는 일 | 유닛 소모 |
|---|---|---|---|
| 0. 사전 점검 | **CPU** | 기존 9개 실험을 v2 파서로 재채점하고 로컬 결과와 비교 | 거의 없음 |
| 1. 최종 생성 | **A100 GPU** | `configs/final_16k.yaml`로 900문항 생성 (약 2시간). 끝나면 런타임 자동 반납 | 약 25~30 (추정) |
| 2. 최종 재채점 | **CPU** | 10개 실행(기존 9개 + 최종)을 v2로 재채점, §5 표 생성 | 거의 없음 |

**왜 이 순서인가**: 최종 config(16k + 기존 프롬프트, presence 1.5)는 9개 실험의 v2 재채점 결과를 보고 정했다. 0단계는 A100을 쓰기 전에 Drive의 outputs와 도구 파일이 제대로 있는지 확인하는 점검이다.

**필요한 파일** (Drive 프로젝트 폴더): `src/mmmu_eval/parse_v2.py`, `scripts/rescore.py`, `scripts/make_report_table.py`, `configs/final_16k.yaml`

## 결과가 저장되는 위치

기준 폴더: `.../최종/final_16k/qwen3_vl_mmmu_eval/qwen3_vl_mmmu_eval/` (아래 `프로젝트/`)

| 무엇 | 경로 | 내용 |
|---|---|---|
| 기존 실험 결과 | `프로젝트/outputs/<실행이름>/` | `scored.jsonl`(문항별 응답·판정), `scores.json`, `run_meta.json` 등. **재채점은 이 폴더를 수정하지 않는다** |
| 최종 실행 결과 | `프로젝트/outputs/final_seed3407_16k/` | 1단계가 생성 |
| 0단계 재채점 (9개 실험) | `프로젝트/reports/rescore_pre/` | `summary.md`, `summary.json`, `audit_sample.jsonl` |
| 2단계 재채점 (10개 실험) | `프로젝트/reports/rescore/` | 같은 3개 파일. 최종 실행과 McNemar 비교 포함 |

- `summary.md`: 실행별 원본/v2 점수, 잘림·unparsed 비율, McNemar 표 (사람이 읽는 용도)
- `summary.json`: 같은 내용, 기계 판독용
- `audit_sample.jsonl`: v2가 복구한 응답 30건과 잘려서 제외한 10건 (수기 검수용)
- 문항별 v2 판정과 §5 30과목 표는 파일로 저장되지 않고 셀 출력으로만 나온다 (보고서에 붙여 넣은 뒤에는 보고서가 기록)

## 단계 0. 사전 점검 (CPU 런타임)
기존 9개 실험을 v2로 재채점한다. 아래 확인 셀에서 모두 `OK`가 나와야 한다. `MISMATCH`가 있으면 폴더 이름이나 파일 버전을 먼저 확인한다 (A100 사용 전에).

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/최종/final_16k/qwen3_vl_mmmu_eval/qwen3_vl_mmmu_eval")
OUTPUTS = DRIVE_PROJECT / "outputs"
DATA_ROOT = Path("/content/drive/MyDrive/mmmu_eval/data")

import os, subprocess, sys, json
OUT_DIR = DRIVE_PROJECT / "reports" / "rescore_pre"

assert (DRIVE_PROJECT / "src/mmmu_eval/parse_v2.py").exists(), "parse_v2.py가 프로젝트에 없습니다"
assert (DRIVE_PROJECT / "scripts/rescore.py").exists(), "scripts/rescore.py가 프로젝트에 없습니다"
print("outputs 안의 실행 폴더:", sorted(p.name for p in OUTPUTS.iterdir() if (p / "scored.jsonl").exists()))

env = dict(os.environ, PYTHONPATH=str(DRIVE_PROJECT / "src"))
r = subprocess.run([sys.executable, "scripts/rescore.py",
                    "--outputs-dir", str(OUTPUTS), "--out-dir", str(OUT_DIR)],
                   cwd=str(DRIVE_PROJECT), env=env, capture_output=True, text=True)
print(r.stdout[-9000:]); print(r.stderr[-3000:])
print("저장 위치:", OUT_DIR)

Mounted at /content/drive
outputs 안의 실행 폴더: ['baseline_seed3407_baseline', 'baseline_seed3407_detailed_prompt', 'baseline_seed3407_presence0', 'baseline_seed3407_prompt', 'baseline_seed42_16k', 'baseline_seed42_4k', 'baseline_seed42_8k', 'baseline_seed42_high_max', 'baseline_seed42_high_min']
runs: ['baseline_seed42_4k', 'baseline_seed42_8k', 'baseline_seed42_16k', 'baseline_seed42_high_max', 'baseline_seed42_high_min', 'baseline_seed3407_baseline', 'baseline_seed3407_prompt', 'baseline_seed3407_detailed_prompt', 'baseline_seed3407_presence0']
# Rescore summary (orig = Qwen rule parser, v2 = orig + conservative fallback)

| run | acc orig | acc v2 | Δ | trunc% | unparsed% orig | unparsed% v2 | fallback used/correct | v2 unparsed (not truncated) |
|---|---|---|---|---|---|---|---|---|
| baseline_seed42_4k | 44.11 | 58.89 | +14.78 | 19.2 | 37.3 | 16.8 | 185/133 | 18 (open 17) |
| baseline_seed42_8k | 46.56 | 61.56 | +15.00 | 14.1 | 33.6 | 11.3 | 200/135 | 20 (open 20) |
| baseline_seed42

아래 셀은 내가 로컬에서 미리 재채점한 결과랑 비교하는 거라 그냥 검증용

In [ ]:
# 로컬에서 미리 확인한 v2 점수와 같은지 비교 (다르면 폴더 이름/파일 버전부터 확인)
EXPECTED = {"baseline_seed3407_baseline": 62.89, "baseline_seed42_8k": 61.56, "baseline_seed42_16k": 64.22,
            "baseline_seed3407_detailed_prompt": 62.11, "baseline_seed3407_presence0": 58.89}
summary = json.load(open(OUT_DIR / "summary.json", encoding="utf-8"))["runs"]
for run, exp in EXPECTED.items():
    got = summary.get(run, {}).get("acc_v2")
    print(("OK       " if got is not None and abs(got - exp) < 0.01 else "MISMATCH "), run, "expected", exp, "got", got)

OK        baseline_seed3407_baseline expected 62.89 got 62.888888888888886
OK        baseline_seed42_8k expected 61.56 got 61.55555555555556
OK        baseline_seed42_16k expected 64.22 got 64.22222222222223
OK        baseline_seed3407_detailed_prompt expected 62.11 got 62.111111111111114
OK        baseline_seed3407_presence0 expected 58.89 got 58.888888888888886


## 단계 1. 최종 생성 (A100 런타임)
런타임을 **A100 GPU**로 바꾼 뒤 실행한다. 셀이 끝나면(성공이든 실패든) 자동으로 런타임을 반납한다.

- 성공 여부는 셀 출력 끝의 `exit code: 0`으로 확인한다.
- 중간에 끊겼다면 같은 셀을 다시 실행한다 (같은 출력 폴더로 이어서 실행되도록 설계돼 있다. 이 동작은 아직 이 노트북에서 검증하지 않았다).
- 시작 후 처음 몇 분은 모델 로딩과 첫 과목 생성이 정상인지 지켜본다.
- `runtime.unassign()`은 되돌릴 수 없다. 같은 세션에서 다른 작업을 더 하려면 마지막 줄을 주석 처리한다.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import shutil
import sys
import os

# ============================================================
# 설정
# ============================================================

DRIVE_PROJECT = Path(
    "/content/drive/MyDrive/Colab Notebooks/최종/final_16k/qwen3_vl_mmmu_eval/qwen3_vl_mmmu_eval"
)

LOCAL_PROJECT = Path("/content/qwen3_vl_mmmu_eval")

DATA_ROOT = Path("/content/drive/MyDrive/mmmu_eval/data")

# False = 과목당 1문항 Smoke Test
# True  = 전체 900문항
RUN_FULL = True


# ============================================================
# 1. Google Drive 연결
# ============================================================

drive.mount("/content/drive")

print("\n[1/6] Google Drive 연결 완료")


# ============================================================
# 2. 프로젝트 확인
# ============================================================

assert DRIVE_PROJECT.exists(), (
    f"프로젝트 폴더가 없습니다:\n{DRIVE_PROJECT}"
)

assert (DRIVE_PROJECT / "requirements.txt").exists(), (
    "requirements.txt 파일이 없습니다."
)

assert (DRIVE_PROJECT / "scripts/run_mmmu_eval.sh").exists(), (
    "scripts/run_mmmu_eval.sh 파일이 없습니다."
)

assert DATA_ROOT.exists(), (
    f"MMMU 데이터 폴더가 없습니다:\n{DATA_ROOT}"
)

print("[2/6] 프로젝트 및 데이터 확인 완료")


# ============================================================
# 3. 프로젝트를 Colab 로컬 /content로 복사
# ============================================================

if LOCAL_PROJECT.exists():
    shutil.rmtree(LOCAL_PROJECT)

shutil.copytree(
    DRIVE_PROJECT,
    LOCAL_PROJECT,
    ignore=shutil.ignore_patterns(
        ".venv",
        "__pycache__",
        "*.pyc",
        "outputs" # outputs 추가
    )
)


print("[3/6] 프로젝트 복사 완료")
print(LOCAL_PROJECT)


# ============================================================
# 4. requirements 설치
# ============================================================

print("\n[4/6] 패키지 설치 시작\n")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(LOCAL_PROJECT / "requirements.txt")
    ],
    check=True
)

print("\n✅ 패키지 설치 완료")


# ============================================================
# 5. pytest
# ============================================================

# print("\n[5/6] 프로젝트 테스트 실행\n")

# subprocess.run(
#     [
#         sys.executable,
#         "-m",
#         "pytest",
#         "tests/",
#         "-q"
#     ],
#     cwd=str(LOCAL_PROJECT),
#     check=True
# )

# print("\n✅ pytest 통과")


# ============================================================
# GPU 확인
# ============================================================

print("\nGPU 확인\n")

subprocess.run(
    ["nvidia-smi"],
    check=True
)


# ============================================================
# 6. MMMU 실행
# ============================================================

env = os.environ.copy()

# 가상환경 대신 현재 Colab Python 사용
env["PYTHON_BIN"] = sys.executable

print("\n현재 사용 Python:")
print(sys.executable)


# ------------------------------------------------------------
# 전체 900문항
# ------------------------------------------------------------

if RUN_FULL:

    OUTPUT_DIR = str(DRIVE_PROJECT / "outputs" / "final_seed3407_16k")   # 변경

    cmd = [
        "bash",
        "scripts/run_mmmu_eval.sh",

        "--model_path",
        "Qwen/Qwen3-VL-4B-Instruct",

        "--model_revision",
        "ebb281ec70b05090aa6165b016eac8ec08e71b17",

        "--data_root",
        str(DATA_ROOT),

        "--config",
        "configs/final_16k.yaml", # 변경

        "--seed",
        "3407",

        "--output_dir",
        OUTPUT_DIR,

        "--require_gpu",
        "A100",

        "--stage",
        "all"
    ]

    print("\n[6/6] 🚀 900문항 전체 평가 시작")


# subprocess.run(
#     cmd,
#     cwd=str(LOCAL_PROJECT),
#     env=env,
#     check=True
# )

env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.Popen(
    cmd,
    cwd=str(LOCAL_PROJECT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"❌ 스크립트 실패 (exit code {proc.returncode}) — 위 로그의 마지막 Traceback 확인")

print("\n" + "=" * 60)
print("✅ 실행 완료")
print("=" * 60)
print("결과 폴더:")
print(OUTPUT_DIR)
print("=" * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[1/6] Google Drive 연결 완료
[2/6] 프로젝트 및 데이터 확인 완료
[3/6] 프로젝트 복사 완료
/content/qwen3_vl_mmmu_eval

[4/6] 패키지 설치 시작


✅ 패키지 설치 완료

GPU 확인


현재 사용 Python:
/usr/bin/python3

[6/6] 🚀 900문항 전체 평가 시작
GPU: NVIDIA A100-SXM4-40GB
2026-09-27 01:23:06.332067: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-27 01:23:06.405278: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Accounting: 30
Agriculture: 

## 단계 2. 최종 재채점 (CPU 런타임)
런타임을 **CPU(GPU 없음)**로 다시 바꿔 실행한다. 1단계에서 런타임이 반납됐으므로 새 세션이 시작된다.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/최종/final_16k/qwen3_vl_mmmu_eval/qwen3_vl_mmmu_eval")
OUTPUTS = DRIVE_PROJECT / "outputs"
DATA_ROOT = Path("/content/drive/MyDrive/mmmu_eval/data")

import os, subprocess, sys, json
OUT_DIR = DRIVE_PROJECT / "reports" / "rescore"

assert (DRIVE_PROJECT / "src/mmmu_eval/parse_v2.py").exists(), "parse_v2.py가 프로젝트에 없습니다"
assert (DRIVE_PROJECT / "scripts/rescore.py").exists(), "scripts/rescore.py가 프로젝트에 없습니다"
print("outputs 안의 실행 폴더:", sorted(p.name for p in OUTPUTS.iterdir() if (p / "scored.jsonl").exists()))

env = dict(os.environ, PYTHONPATH=str(DRIVE_PROJECT / "src"))
r = subprocess.run([sys.executable, "scripts/rescore.py",
                    "--outputs-dir", str(OUTPUTS), "--out-dir", str(OUT_DIR)],
                   cwd=str(DRIVE_PROJECT), env=env, capture_output=True, text=True)
print(r.stdout[-9000:]); print(r.stderr[-3000:])
print("저장 위치:", OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
outputs 안의 실행 폴더: ['baseline_seed3407_baseline', 'baseline_seed3407_detailed_prompt', 'baseline_seed3407_presence0', 'baseline_seed3407_prompt', 'baseline_seed42_16k', 'baseline_seed42_4k', 'baseline_seed42_8k', 'baseline_seed42_high_max', 'baseline_seed42_high_min', 'final_seed3407_16k']
runs: ['baseline_seed42_4k', 'baseline_seed42_8k', 'baseline_seed42_16k', 'baseline_seed42_high_max', 'baseline_seed42_high_min', 'baseline_seed3407_baseline', 'baseline_seed3407_prompt', 'baseline_seed3407_detailed_prompt', 'baseline_seed3407_presence0', 'final_seed3407_16k']
# Rescore summary (orig = Qwen rule parser, v2 = orig + conservative fallback)

| run | acc orig | acc v2 | Δ | trunc% | unparsed% orig | unparsed% v2 | fallback used/correct | v2 unparsed (not truncated) |
|---|---|---|---|---|---|---|---|---|
| baseline_seed42_4k | 44.11 | 58.89 | +14.78 | 19.2 | 37.

### 보고서 §5 표 생성
`Overall (macro avg)`의 v2/원본 값이 보고서 §5, §6에 들어가고, 아래 출력의 GPU·VRAM·소요 시간·잘림 수치는 §1과 §7에 들어간다. `reports/mmmu_baseline.md`의 `TODO`를 이 값으로 교체한다.

In [ ]:
import os, subprocess, sys
env = dict(os.environ, PYTHONPATH=str(DRIVE_PROJECT / "src"))
r = subprocess.run([sys.executable, "scripts/make_report_table.py", str(OUTPUTS / "final_seed3407_16k")],
                   cwd=str(DRIVE_PROJECT), env=env, capture_output=True, text=True)
print(r.stdout); print(r.stderr[-3000:])

| No. | Subject | Data Num | Acc (v2) | Acc (Qwen 원본 규칙) |
|---|---|---|---|---|
| 1 | Accounting | 30 | 76.67 | 63.33 |
| 2 | Agriculture | 30 | 50.00 | 43.33 |
| 3 | Architecture_and_Engineering | 30 | 50.00 | 40.00 |
| 4 | Art | 30 | 56.67 | 46.67 |
| 5 | Art_Theory | 30 | 83.33 | 60.00 |
| 6 | Basic_Medical_Science | 30 | 76.67 | 63.33 |
| 7 | Biology | 30 | 50.00 | 43.33 |
| 8 | Chemistry | 30 | 50.00 | 36.67 |
| 9 | Clinical_Medicine | 30 | 70.00 | 50.00 |
| 10 | Computer_Science | 30 | 56.67 | 40.00 |
| 11 | Design | 30 | 80.00 | 70.00 |
| 12 | Diagnostics_and_Laboratory_Medicine | 30 | 30.00 | 16.67 |
| 13 | Economics | 30 | 86.67 | 70.00 |
| 14 | Electronics | 30 | 66.67 | 56.67 |
| 15 | Energy_and_Power | 30 | 53.33 | 30.00 |
| 16 | Finance | 30 | 63.33 | 63.33 |
| 17 | Geography | 30 | 56.67 | 40.00 |
| 18 | History | 30 | 66.67 | 36.67 |
| 19 | Literature | 30 | 83.33 | 50.00 |
| 20 | Manage | 30 | 56.67 | 43.33 |
| 21 | Marketing | 30 | 90.00 | 80.00 |
| 22 | Materials | 3